In [1]:
import json
with open('../jsons/all_songs.json', 'r') as f:
    all_songs = json.load(f)

In [2]:
all_songs.keys()

dict_keys(['2689', '2771', '2635', '2805', '2769', '8811', '5890', '8634', '2548', '2710', '5889', '4983', '2688', '7110', '2638', '2701', '2537', '7111', '2690', '2552', '8630', '5891', '8820', '7796', '7167', '5887', '2704', '7817', '2588', '2750', '7814', '8681', '2647', '2655', '4702', '2661', '8810', '2729', '5530', '2757', '2700', '2696', '2637', '2653', '2813', '8636', '2752', '2558', '2580', '2755', '6068', '8678', '2745', '7797', '7794', '6317', '2775', '8682', '6655', '7807', '2564', '6932', '6321', '4562', '2686', '7767', '2717', '7816', '2810', '4560', '6319', '2712', '2659', '2545', '2600', '2622', '4566', '2559', '2758', '2569', '8639', '7785', '2730', '2582', '7758', '2812', '2679', '2528', '2746', '8628', '2554', '6657', '2682', '6942', '5627', '2522', '2687', '2598', '2770', '7778', '2784', '2565', '5714', '2642', '2660', '7791', '8629', '7810', '5591', '6243', '5529', '4703', '7775', '5532', '2541', '2776', '6659', '2709', '2657', '2563', '8821', '5594', '2780', '2803

In [81]:
all_songs['2689']

[[-1.2100567817687988,
  0.2913461923599243,
  -4.815102577209473,
  3.9990382194519043,
  -0.9506726861000061,
  2.9242568016052246,
  0.7342255115509033,
  6.335336208343506,
  -2.1053762435913086,
  -0.29092493653297424,
  7.168038845062256,
  7.045341491699219,
  -3.3399763107299805,
  3.7533600330352783,
  0.6485605239868164,
  -0.3182080388069153,
  3.8603515625,
  -0.032119348645210266,
  -1.8644012212753296,
  -0.43043577671051025,
  -0.3856046497821808,
  8.181563377380371,
  0.2934073507785797,
  -0.44122883677482605,
  -1.8552765846252441,
  7.984591484069824,
  4.941964149475098,
  0.19465546309947968,
  -0.6366775631904602,
  -0.011952435597777367,
  0.0821729451417923,
  -2.1306240558624268,
  0.26666590571403503,
  -1.5171658992767334,
  0.5576554536819458,
  2.279040575027466,
  1.9140775203704834,
  3.026931047439575,
  -0.6502913236618042,
  10.816767692565918,
  -0.33408504724502563,
  -0.31441906094551086,
  -0.7316544055938721,
  7.654202461242676,
  -0.12373538315

In [59]:
import sys
sys.path.append("/home/youssef/Documents/python/music-map/backend")
from data_extraction.similarity_model import *
all_tracks = load_all_tracks("/home/youssef/Documents/python/music-map/backend/data_extraction/music.duckdb")
X_pca, pca, scaler, meta = reduce_features(all_tracks, meta_cols=["id", "artist", "artist_id"])
concated = pd.DataFrame(X_pca).join(meta)


Using 224 PCA components to explain 95% of variance (reduced from 4377 original dimensions)


In [60]:
combined_dict = {}
for i, item in enumerate(concated['id']):
    mert_vec = np.array(all_songs[str(item)][5]) / np.linalg.norm(all_songs[str(item)][5])       # or z-score
    essentia_vec = X_pca[i] / np.linalg.norm(X_pca[i])
    combined = np.concatenate([mert_vec, essentia_vec])
    combined_dict[i] = combined
combined_df = pd.DataFrame(combined_dict).T.join(meta)

In [91]:
import numpy as np
from itertools import combinations, product
import tqdm

intra = {}
unique_ids = list(set(item for item in combined_df['artist_id']))
for id in tqdm.tqdm(unique_ids):
    artist_songs = [
        combined_df.iloc[i, :-3].to_numpy() 
        for i in range(combined_df.shape[0]) 
        if id == combined_df.iloc[i]['artist_id']
    ]
    similarities = get_intra_similarity(artist_songs)
    intra[id] = similarities
with open("../jsons/hybrid_intra.json", "w") as f:
  json.dump(intra, f)

100%|██████████| 100/100 [00:26<00:00,  3.84it/s]


In [84]:
import json
from itertools import combinations
import tqdm
from collections import defaultdict

inter = defaultdict(dict)
combs = [(a, b) for a, b in combinations(unique_ids, 2)]
for artist1, artist2 in tqdm.tqdm(combs):
  artist1_songs = [combined_df.iloc[song, :-3].to_numpy() for i, song in enumerate(range(combined_df.shape[0])) if artist1 == combined_df.iloc[i]['artist_id']]
  artist2_songs = [combined_df.iloc[song, :-3].to_numpy() for i, song in enumerate(range(combined_df.shape[0])) if artist2 == combined_df.iloc[i]['artist_id']]
  similarities = get_inter_similarity(artist1_songs, artist2_songs)
  inter[artist1][artist2] = similarities
with open("../jsons/hybrid_inter.json", "w") as f:
  json.dump(inter, f)

100%|██████████| 4950/4950 [52:26<00:00,  1.57it/s]  


In [17]:
import json
import numpy as np
with open("../jsons/hybrid_intra.json", "r") as f:
    intra = json.load(f)

intra_sims = []
for artist_id, value in intra.items():
    intra_sims += value

import json
with open("../jsons/hybrid_inter.json", "r") as f:
    inter = json.load(f)

inter_sims = []
for artist_id, value in inter.items():
    for artist_id2, value2 in value.items():
        inter_sims +=value2

In [43]:
import numpy as np
def cohens_d(intra_sims, inter_sims):
    n1, n2 = len(intra_sims), len(inter_sims)
    var1, var2 = np.var(intra_sims, ddof=1), np.var(inter_sims, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(intra_sims) - np.mean(inter_sims)) / pooled_std

from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
def get_intra_similarity(song_outputs):
  similarities = []
  for song1, song2 in combinations(song_outputs, 2):
    song1 = np.array(song1).reshape(1, -1)
    song2 = np.array(song2).reshape(1, -1)
    similarities.append(cosine_similarity(song1, song2).item())
  return similarities

from itertools import product
def get_inter_similarity(outputs1, outputs2):
  similarities = []
  for out1, out2 in product(outputs1, outputs2):
    out1 = np.array(out1).reshape(1, -1)
    out2 = np.array(out2).reshape(1, -1)
    similarities.append(cosine_similarity(out1, out2).item())
  return similarities

In [93]:
cohens_d(intra_sims, inter_sims)

np.float64(1.516450487448376)

In [18]:
import numpy as np
avgs = {}
medians = {}
for key, value in inter.items():
    avgs[key] = {}
    medians[key] = {}
    for other, sim in value.items():
        avgs[key][other] = np.mean(sim).item()

        medians[key][other] = np.median(sim).item()

with open("../jsons/hybrid_medians.json", "w") as f:
    json.dump(medians, f)
with open("../jsons/hybrid_avgs.json", "w") as f:
    json.dump(avgs, f)
# avgs

In [66]:
avgs

{'1': {'2': 0.4643669599972919,
  '3': 0.4566046377497686,
  '4': 0.47600219220616424,
  '5': 0.4759443481602848,
  '6': 0.44637086808677223,
  '7': 0.47380401041345543,
  '8': 0.46554720452380494,
  '9': 0.46761725589882397,
  '10': 0.4830562188365103,
  '11': 0.47521549931897666,
  '12': 0.4969431876725089,
  '13': 0.4774714408829525,
  '14': 0.4712176881804239,
  '15': 0.48389597341917195,
  '16': 0.4647413458528792,
  '17': 0.46419710998473207,
  '18': 0.4757648215657828,
  '19': 0.48675825251046356,
  '21': 0.4733238520143803,
  '22': 0.48767853207502576,
  '23': 0.47715787092571593,
  '24': 0.4649784908706314,
  '25': 0.48818584334475806,
  '26': 0.4750489466266557,
  '27': 0.49003118098800913,
  '28': 0.4759590426437875,
  '29': 0.4745650522391521,
  '31': 0.48092585552923645,
  '32': 0.46384317116708956,
  '33': 0.4800445067452169,
  '34': 0.4773377541598801,
  '36': 0.47284051922901527,
  '37': 0.47225072346410607,
  '39': 0.4777882591211691,
  '40': 0.46846618718704824,
  '41

In [67]:
avgs['1']['128']

0.4727828535052415

In [ ]:
# full two way dict with names and similarity
unique_ids = list(set(item for item in combined_df['artist_id']))
from copy import deepcopy
from itertools import combinations
full_avgs = deepcopy(avgs)
full_avgs['128'] = {}
for a, b in combinations(unique_ids, 2):
    full_avgs[str(b)][str(a)] = avgs[str(a)][str(b)]
full_avgs

with open('../jsons/full_avgs_essentiaxmert.json', 'w') as f:
    json.dump(full_avgs, f)

In [44]:
with open("../jsons/hybrid_avgs.json", "r") as f:
    avgs = json.load(f)
from collections import defaultdict
import sys
import os
# sys.path.append("/home/youssef/Documents/python/music-map/backend")
from data_extraction.similarity_model import get_names, lookup_artist
lookup_table = dict(get_names())
inter_names = defaultdict(dict)
for id, value in avgs.items():
    for other_id, sim in value.items():
        inter_names[id][other_id] = sim 
        inter_names[other_id][id] = sim
inter_names
# with open("../jsons/hybrid_named.json", "w") as f:
#     json.dump(inter_names, f)

defaultdict(dict,
            {'1': {'2': 0.4643669599972919,
              '3': 0.4566046377497686,
              '4': 0.47600219220616424,
              '5': 0.4759443481602848,
              '6': 0.44637086808677223,
              '7': 0.47380401041345543,
              '8': 0.46554720452380494,
              '9': 0.46761725589882397,
              '10': 0.4830562188365103,
              '11': 0.47521549931897666,
              '12': 0.4969431876725089,
              '13': 0.4774714408829525,
              '14': 0.4712176881804239,
              '15': 0.48389597341917195,
              '16': 0.4647413458528792,
              '17': 0.46419710998473207,
              '18': 0.4757648215657828,
              '19': 0.48675825251046356,
              '21': 0.4733238520143803,
              '22': 0.48767853207502576,
              '23': 0.47715787092571593,
              '24': 0.4649784908706314,
              '25': 0.48818584334475806,
              '26': 0.4750489466266557,
             

In [72]:
from data_extraction.similarity_model import get_names, lookup_artist
lookup_table = dict(get_names())
inter_names = defaultdict(dict)
for id, value in full_avgs.items():
    for other_id, sim in value.items():
        name = lookup_artist(id)
        othername = lookup_artist(other_id)
        inter_names[name][othername] = sim 
inter_names
with open("../jsons/full_avgs_essentiaxmert.json", "w") as f:
    json.dump(inter_names, f)

In [78]:
import json

# with open('../jsons/hybrid_named.json', 'r') as f:
#     data = json.load(f)
# with open('../jsons/artistNames.json', 'r') as f:
#     reversed = json.load(f)

sorted(inter_names['عمرو دياب'].items(), key = lambda x: x[1], reverse=True) 

[('رامي صبري', 0.4969431876725089),
 ('خالد سليم', 0.4939009336235615),
 ('صابر الرباعي', 0.4921014060809153),
 ('محمد محي', 0.4913166973520902),
 ('زكي ناصيف', 0.4908739671193229),
 ('محمد فؤاد', 0.49003118098800913),
 ('هشام عباس', 0.48818584334475806),
 ('حمزة نمرة', 0.48767853207502576),
 ('راغب علامة', 0.48675825251046356),
 ('بهاء سلطان', 0.48389597341917195),
 ('خالد', 0.4830562188365103),
 ('عمرو مصطفى', 0.4828521200168659),
 ('سميرة سعيد', 0.48201817179606155),
 ('Sharmoofers', 0.4815231941176395),
 ('إيهاب توفيق', 0.48092585552923645),
 ('هاني شاكر', 0.4802534989478921),
 ('حمو بيكا', 0.4800445067452169),
 ('محمد رمضان', 0.4786079784917425),
 ('وائل جسار', 0.47792497771821457),
 ('رعد و ميثاق', 0.47787696972087795),
 ('ملحم زين', 0.4777882591211691),
 ('عاصي الحلاني', 0.4776824721229829),
 ('محمد السالم', 0.47761642855094466),
 ('تامر عاشور', 0.4774714408829525),
 ('فضل شاكر', 0.4773377541598801),
 ('وائل كفوري', 0.4772918399806521),
 ('زياد الرحباني', 0.477262317400621),
 ('

In [46]:
inter_names

defaultdict(dict, {})